In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.nn as pyg_nn
from pettingzoo.mpe import simple_tag_v2
import os

# Initialize the environment
env = simple_tag_v2.parallel_env(render_mode=None, num_adversaries=4, num_good=1, num_obstacles=2)
env.reset()

# Parameters
num_class_a = 3
num_class_b = 1
num_adversaries = 4
num_agents = 1  # Only one agent being chased by adversaries
num_obstacles = 2
adversary_agents = [agent for agent in env.agents if 'adversary' in agent]
good_agents = [agent for agent in env.agents if 'agent' in agent]

print("Adversary Agents:", adversary_agents)
print("Good Agents:", good_agents)


Adversary Agents: ['adversary_0', 'adversary_1', 'adversary_2', 'adversary_3']
Good Agents: ['agent_0']


### Define Embedding and CGN layer

In [2]:
# Define linear embedding layer
class LinearEmbedding(nn.Module):
    def __init__(self, input_dim, embed_dim):
        super(LinearEmbedding, self).__init__()
        self.linear = nn.Linear(input_dim, embed_dim)
    
    def forward(self, x):
        return self.linear(x)

# Define GCN layer
class GCNLayer(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(GCNLayer, self).__init__()
        self.conv1 = pyg_nn.GCNConv(input_dim, output_dim)
    
    def forward(self, x, edge_index):
        return self.conv1(x, edge_index)

# Parameters for embedding and GCN
communication_range = 1.5  # Define the communication range
embedding_dim = 8          # Dimension for linear embedding
gcn_output_dim = 16        # Dimension of GCN output

# Initialize embedding and GCN layers
embedding_layer = LinearEmbedding(input_dim=5, embed_dim=embedding_dim)  # input_dim includes the class identifier
gcn_layer = GCNLayer(input_dim=embedding_dim, output_dim=gcn_output_dim)


### Encoder-Decoder

### Sequence to Sequence Feature Transformation in PyTorch

In sequence-to-sequence models, each time step of the input sequence is represented by a feature vector. If the feature dimensionality (`input_size`) of the sequence at each time step differs from the hidden state dimensionality (`hidden_size`) expected by an LSTM (or other recurrent layers), a linear projection can be applied to map the input features to the desired size.

#### 1. **Sequence Representation**

Consider an input sequence $ X $ of length $ T $, where each time step $ t $ is represented by a feature vector $ x_t $ of size $ d_{\text{input}} $:

$
X = [x_1, x_2, \dots, x_T], \quad x_t \in \mathbb{R}^{d_{\text{input}}}
$

For a batch of sequences of size $ B $, the input is represented as a tensor of shape:

$
\text{input\_seq} \in \mathbb{R}^{B \times T \times d_{\text{input}}}
$

#### 2. **LSTM Hidden Size**

An LSTM expects the dimensionality of the input features to match its hidden state size $ d_{\text{hidden}} $. If $ d_{\text{input}} \neq d_{\text{hidden}} $, we need to adjust the feature dimensions for each time step using a linear transformation.

#### 3. **Linear Transformation**

The linear transformation adjusts the feature size:

$
y_t = W x_t + b
$

However, in practice, the multiplication is performed with $ W $ transposed:

$
y_t = x_t W^T + b
$

Where:
- $ W \in \mathbb{R}^{d_{\text{hidden}} \times d_{\text{input}}} $ is the weight matrix, transposed for multiplication.
- $ b \in \mathbb{R}^{d_{\text{hidden}}} $ is the bias term.
- $ x_t \in \mathbb{R}^{d_{\text{input}}} $ is the feature vector at time step $ t $.

This operation projects each feature vector $ x_t $ from $ \mathbb{R}^{d_{\text{input}}} $ to a new feature space $ \mathbb{R}^{d_{\text{hidden}}} $, ensuring that the LSTM receives input vectors of the correct dimensionality.

The new sequence after transformation has the shape:

$
\text{input\_seq'} \in \mathbb{R}^{B \times T \times d_{\text{hidden}}}
$

#### 4. **Code Example**

In PyTorch, this is implemented using `torch.matmul` to perform matrix multiplication, ensuring $ W $ is transposed correctly:

```python
import torch

# Define input sequence with shape (batch_size, seq_length, input_size)
X = torch.tensor([[1.0, 2.0, 3.0], 
                  [4.0, 5.0, 6.0]])  # Shape: (2, 3)

# Define weight matrix W and bias b
W = torch.tensor([[0.2, 0.5, 0.1], 
                  [0.6, 0.3, 0.4]])   # Shape: (2, 3) -> Transposed to (3, 2)

b = torch.tensor([0.1, 0.2])          # Shape: (2)

# Perform the linear transformation for each time step (X W^T + b)
X_proj = torch.matmul(X, W.T) + b
print(X_proj)

### Understanding the Linear Transformation in Sequence-to-Sequence Models

In the context of transforming an input sequence using a linear layer, it is crucial to understand how matrix multiplication works, especially when dealing with feature dimensions that may not align.

#### Matrix Multiplication

Consider an input sequence $ X $ of shape $ (B, T, d_{\text{input}}) $, where:
- $ B $ is the batch size,
- $ T $ is the sequence length,
- $ d_{\text{input}} $ is the dimensionality of each input feature vector.

When we apply a linear transformation to each feature vector $ x_t $ at each time step $ t $ using a weight matrix $ W $ and bias $ b $, the operation can be expressed as:

$
y_t = W x_t + b
$

However, for matrix multiplication to work correctly, we need to ensure the dimensions align. This typically means the weight matrix $ W $ needs to be transposed during multiplication.

#### Transposed Multiplication

If $ W $ has the shape $ (d_{\text{hidden}}, d_{\text{input}}) $, then the multiplication is performed as follows:

$
y_t = x_t W^T + b
$

Here:
- $ x_t $ is a vector of shape $ (d_{\text{input}}) $.
- $ W^T $ has a shape of $ (d_{\text{input}}, d_{\text{hidden}}) $.

This means:
- The input feature vector $ x_t $ of shape $ (1, d_{\text{input}}) $ can be multiplied with $ W^T $ of shape $ (d_{\text{input}}, d_{\text{hidden}}) $.
- The resulting $ y_t $ will have a shape of $ (1, d_{\text{hidden}}) $, as desired.

#### Example with Shapes

Suppose we have:
- $ X \in \mathbb{R}^{2 \times 3} $: A batch of 2 sequences, each with 3 features.
- $ W \in \mathbb{R}^{2 \times 3} $: The weight matrix intended to project input features to a hidden state of size 2.

In this case, $ W $ must be transposed to $ W^T \in \mathbb{R}^{3 \times 2} $ for the multiplication to work. Thus, the operation becomes:

$
X W^T \in \mathbb{R}^{2 \times 3} \cdot \mathbb{R}^{3 \times 2} \rightarrow \mathbb{R}^{2 \times 2}
$

Where:
- The resulting output shape $ \mathbb{R}^{2 \times 2} $ corresponds to the transformed feature vectors for each sequence.

### Conclusion

This method ensures that each feature vector from the input sequence can be transformed correctly, allowing the LSTM to receive input in the expected dimensionality. The use of transposed weights is a common practice in neural network implementations to ensure proper alignment during matrix multiplication.


In [3]:
class Encoder(nn.Module):
    def __init__(self, hidden_size):
        super(Encoder, self).__init__()
        self.hidden_size = hidden_size
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
    
    def forward(self, input_seq, input_lengths):
        # Pack the padded batch of sequences
        packed_input = nn.utils.rnn.pack_padded_sequence(input_seq, input_lengths, batch_first=True, enforce_sorted=False)
        
        # Initialize hidden and cell state with zeros
        h0 = torch.zeros(1, input_seq.size(0), self.hidden_size).to(input_seq.device)
        c0 = torch.zeros(1, input_seq.size(0), self.hidden_size).to(input_seq.device)

        # Forward pass through LSTM
        packed_output, (hidden, _) = self.lstm(packed_input, (h0, c0))
        
        # Hidden state contains the context vector
        return hidden

class Decoder(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(Decoder, self).__init__()
        self.lstm = nn.LSTM(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, context_vector, output_length):
        # Repeat the context vector for the required output length
        decoder_input = context_vector.repeat(output_length, 1, 1).permute(1, 0, 2)
        decoder_output, _ = self.lstm(decoder_input)
        output_seq = self.fc(decoder_output)
        return output_seq

class Seq2Seq(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(Seq2Seq, self).__init__()
        self.encoder = Encoder(hidden_size)
        self.decoder = Decoder(hidden_size, output_size)

    def forward(self, input_seq, input_lengths, output_length):
        # Encode the input sequence
        context_vector = self.encoder(input_seq, input_lengths)
        # Decode to a fixed-length output sequence
        output_seq = self.decoder(context_vector, output_length)
        return output_seq

# Example: Loop generating batches with varying sequence lengths
hidden_size = 64  # Hidden state size
output_size = 1  # Fixed output sequence dimension before 10, but it will create an output of size Batch x output_size x ouput_length
output_length = 40  # The length of the output sequence

model = Seq2Seq(hidden_size, output_size)

#output_decoder_dim = 18
input_actor_network_max_dim = output_length #40 # Temporary solution with padding

### Observation Wrapper

In [4]:
def adversary_observation_wrapper(observations, num_class_A, num_class_B, adversary_agents, num_adversaries, num_agents, num_obstacles):
    assert num_class_A + num_class_B == num_adversaries, "Number of agents assigned to Class A and Class B must match the total number of adversaries."
    
    class_a_agents = adversary_agents[:num_class_A]
    class_b_agents = adversary_agents[num_class_A:num_class_A + num_class_B]
    
    updated_observations = {}
    adversary_positions = {}  # Store adversary positions for communication
    
    # Step 1: Add agent class identifier into observation
    for agent, obs in observations.items():
        # If the agent is an adversary (either Class A or Class B)
        if agent in class_a_agents or agent in class_b_agents:
            agent_class = 0 if agent in class_a_agents else 1  # Class A: 0, Class B: 1
            updated_obs = np.concatenate([obs, [agent_class]])  # Add class identifier
        else:
            updated_obs = obs  # Non-adversary agents keep their observation
        updated_observations[agent] = updated_obs
    
    # Step 2: Gather positions for communication
    for agent in adversary_agents:
        position = updated_observations[agent][2:4]  # Assume position is at index 2:4
        adversary_positions[agent] = position
    
    # Step 3: Apply linear embedding and handle communication within range
    embedded_information = {}
    for agent, obs in updated_observations.items():
        if agent in adversary_agents:
            # Convert observation to torch tensor
            obs_tensor = torch.tensor(obs[:5], dtype=torch.float32)  # Include the class identifier
            embedded_obs = embedding_layer(obs_tensor)  # Apply embedding to observation
            embedded_information[agent] = embedded_obs
    
    # Step 4: Communication and gather data for GCN input
    node_features = []
    edge_index = []
    num_agents_in_graph = len(adversary_agents)
    
    agent_to_idx = {agent: idx for idx, agent in enumerate(adversary_agents)}
    
    for agent in adversary_agents:
        own_position = adversary_positions[agent]
        node_features.append(embedded_information[agent])
        
        for other_agent in adversary_agents:
            if agent != other_agent:
                other_position = adversary_positions[other_agent]
                distance = np.linalg.norm(own_position - other_position)
                
                if distance <= communication_range:  # Check if within communication range
                    edge_index.append([agent_to_idx[agent], agent_to_idx[other_agent]])  # Add edge

    # Convert to torch tensors
    if edge_index:
        edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    else:
        edge_index = torch.empty((2, 0), dtype=torch.long)
    
    node_features = torch.stack(node_features)
    
    # Step 5: Pass the gathered information into a GCN
    gcn_output = gcn_layer(node_features, edge_index)
    
    # Step 6: Concatenate GCN output with agent's own observation
    final_inputs = {}
    for agent_idx, agent in enumerate(adversary_agents):
        own_obs = torch.tensor(updated_observations[agent], dtype=torch.float32)
        final_input = torch.cat([own_obs, gcn_output[agent_idx]])
        final_inputs[agent] = final_input  # This will be used as input to the Actor network
    
    return updated_observations, final_inputs  # Return the updated observations and inputs for Actor network


### Initialize the Environment and Run a Step

In [5]:
# Initialize environment
env.reset()

# Sample action spaces for all agents
actions = {agent: env.action_space(agent).sample() for agent in env.agents}

# Step through the environment
observations, rewards, terminations, truncations, infos = env.step(actions)

# Apply the observation wrapper for adversaries
observations, final_inputs = adversary_observation_wrapper(
    observations, num_class_a, num_class_b, adversary_agents, num_adversaries, num_agents, num_obstacles)

print("Updated Observations:")
for agent, obs in observations.items():
    print(f"{agent}: {obs}")

print("\nFinal Inputs for Actor Network:")
for agent, inp in final_inputs.items():
    print(f"{agent}: {inp}")

Updated Observations:
adversary_0: [ 0.         -0.          0.04569077 -0.04607532 -0.8308723  -0.03935546
 -0.43021861  0.59966457  0.53372282  0.40609357  0.59780043 -0.55192775
 -0.35716015  0.83367527  0.90584916  0.23527817  0.40000001 -0.
  0.        ]
adversary_1: [-0.30000001  0.          0.57941359  0.36001825 -1.36459517 -0.44544902
 -0.96394145  0.19357099 -0.53372282 -0.40609357  0.06407761 -0.95802128
 -0.89088303  0.42758167  0.37212631 -0.17081539  0.40000001 -0.
  0.        ]
adversary_2: [ 0.         -0.          0.64349121 -0.59800303 -1.42867279  0.51257229
 -1.02801907  1.15159225 -0.59780043  0.55192775 -0.06407761  0.95802128
 -0.95496058  1.38560295  0.3080487   0.78720587  0.40000001 -0.
  0.        ]
adversary_3: [ 0.26337501  0.96469355 -0.31146938  0.78759992 -0.47371215 -0.87303072
 -0.07305843 -0.2340107   0.35716015 -0.83367527  0.89088303 -0.42758167
  0.95496058 -1.38560295  1.26300931 -0.59839708  0.40000001 -0.
  1.        ]
agent_0: [ 0.4        -0. 

### Actor Network


In [6]:
class ActorNetwork(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(ActorNetwork, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(64, output_dim)  # Output dimension should match the action space
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Initialize Actor networks for adversaries
actor_networks = {}
for agent in adversary_agents:
    input_dim = len(env.observation_space(agent).low) + 1 + gcn_output_dim  # Observation length plus class id plus GCN output
    output_dim = env.action_space(agent).n  # Assuming discrete action space
    print(f'old agent:{agent} input_dim:{input_dim} output_dim:{output_dim}')
    #input_dim = max(input_dim, input_actor_network_max_dim)
    input_dim = output_length
    print(f'new agent:{agent} input_dim:{input_dim} output_dim:{output_dim}')
    actor_networks[agent] = ActorNetwork(input_dim, output_dim)


old agent:adversary_0 input_dim:35 output_dim:5
new agent:adversary_0 input_dim:40 output_dim:5
old agent:adversary_1 input_dim:35 output_dim:5
new agent:adversary_1 input_dim:40 output_dim:5
old agent:adversary_2 input_dim:35 output_dim:5
new agent:adversary_2 input_dim:40 output_dim:5
old agent:adversary_3 input_dim:35 output_dim:5
new agent:adversary_3 input_dim:40 output_dim:5


### Training Loop

In [8]:
# Training parameters
num_episodes = 100  # Total number of episodes to run
print_interval = 10  # Print rewards every 10 episodes

# Initialize reward tracking
episode_rewards = []
cumulative_reward = 0

# Optimizers for Actor networks and GCN (assuming we are training them)
learning_rate = 0.001
actor_optimizers = {agent: torch.optim.Adam(actor_networks[agent].parameters(), lr=learning_rate) for agent in adversary_agents}
gcn_optimizer = torch.optim.Adam(gcn_layer.parameters(), lr=learning_rate)

# Loss function (placeholder, you need to define based on your RL algorithm)
loss_fn = nn.MSELoss()

# Training loop
for episode in range(1, num_episodes + 1):
    observations = env.reset()
    done = False
    cumulative_reward = 0  # Reset cumulative reward for the episode
    
    while not done:
        # Apply the observation wrapper for adversaries
        observations, final_inputs = adversary_observation_wrapper(
            observations, num_class_a, num_class_b, adversary_agents, num_adversaries, num_agents, num_obstacles)
        actions = {}
        # print(f'env.agents:{env.agents}')
        for agent in env.agents:
            # print(f'agent:{agent}')
            if agent in adversary_agents:
                #print(f'final_inputs:{final_inputs[agent].shape}')
                # Padding
                # m = nn.ConstantPad1d((0, input_actor_network_max_dim - final_inputs[agent].shape[0]), 0)
                # final_inputs_pad = m(final_inputs[agent])
                # Seq2Seq
                # If the input size is different from the hidden size, project the input
                input_size = 1 
                input_seq = final_inputs[agent] # expected Batch x Num Sequences x Sequence Length
                input_seq = input_seq.unsqueeze(1).unsqueeze(0)
                #print(f'input_seq:{input_seq.shape}')
                seq_lengths = [final_inputs[agent].shape[0]]
                if input_size != hidden_size:
                    input_seq = F.linear(input_seq, torch.randn(hidden_size, input_size))

                # Forward pass
                final_inputs_pad = model(input_seq, seq_lengths, output_length)
                #print(f'final_inputs_pad:{final_inputs_pad.shape}')
                final_inputs_pad = final_inputs_pad.squeeze(0).squeeze(1)
                #print(f'final_inputs_pad:{final_inputs_pad.shape}')

                # Get the input for the Actor network
                actor_input = final_inputs_pad #final_inputs[agent]
                # Get action probabilities (assuming discrete action space)
                action_probs = actor_networks[agent](actor_input)
                # Sample an action (for simplicity, we take the action with the highest probability)
                action = torch.argmax(action_probs).item()
                actions[agent] = action
            else:
                # For non-adversary agents, sample random actions
                actions[agent] = env.action_space(agent).sample()
        
        # Step the environment
        next_observations, rewards, terminations, truncations, infos = env.step(actions)
        
        # Update cumulative reward
        cumulative_reward += sum(rewards.values())
        
        # Placeholder for training step (you need to implement your RL algorithm here)
        # For example, compute loss and update networks
        
        # For simplicity, let's assume we have a target value (dummy value here)
        target = torch.zeros(1)
        loss = 0
        for agent in adversary_agents:
            # print(f'final_inputs2:{final_inputs[agent].shape}')
            m = nn.ConstantPad1d((0, input_actor_network_max_dim - final_inputs[agent].shape[0]), 0)
            final_inputs_pad = m(final_inputs[agent])
            # print(f'final_inputs_pad2:{final_inputs_pad.shape}')
            # Get the predicted value
            actor_input = final_inputs_pad # final_inputs[agent]
            prediction = actor_networks[agent](actor_input)
            # Compute loss (this is a placeholder)
            loss += loss_fn(prediction.unsqueeze(0), target)
        
        # Backpropagation
        gcn_optimizer.zero_grad()
        for optimizer in actor_optimizers.values():
            optimizer.zero_grad()
        
        loss.backward()
        
        gcn_optimizer.step()
        for optimizer in actor_optimizers.values():
            optimizer.step()
        
        # Update observations
        observations = next_observations
        
        # Check if all agents are done
        done = all(terminations.values()) or all(truncations.values())
    
    # Append cumulative reward for the episode
    episode_rewards.append(cumulative_reward)
    
    # Print rewards every 'print_interval' episodes
    if episode % print_interval == 0:
        avg_reward = sum(episode_rewards[-print_interval:]) / print_interval
        print(f"Episode {episode}: Average Reward: {avg_reward}")


Episode 10: Average Reward: -12.874717736075842
Episode 20: Average Reward: -8.780518231813335
Episode 30: Average Reward: 10.177474049901644
Episode 40: Average Reward: 12.638981103371124
Episode 50: Average Reward: -3.1102894132967878
Episode 60: Average Reward: -0.8357273355793495
Episode 70: Average Reward: -13.234783875918172
Episode 80: Average Reward: -26.241090607965663
Episode 90: Average Reward: 19.26531841023889
Episode 100: Average Reward: -4.5152983656593575


### Save the Models

In [9]:
# Create a directory to save models
model_dir = 'saved_models'
if not os.path.exists(model_dir):
    os.makedirs(model_dir)


def save_models(num_class_A, num_class_B, adversary_agents):
    class_a_agents = adversary_agents[:num_class_A]
    class_b_agents = adversary_agents[num_class_A:num_class_A + num_class_B]
    print(f'ctype:{class_a_agents}')
    print(f'ctype:{class_b_agents}')
    for agent in class_a_agents:
        if agent in class_a_agents:
            print(f"agent A:{agent}")
            torch.save(actor_networks[agent].state_dict(), os.path.join(model_dir, f"actor_class_A.pth"))
            break
    for agent in class_b_agents:
        if agent in class_b_agents:
            print(f"agent B:{agent}")
            torch.save(actor_networks[agent].state_dict(), os.path.join(model_dir, f"actor_class_B.pth"))
            break

    #torch.save(class_a_agents[0].state_dict(), os.path.join(model_dir, f"actor_class_A.pth"))
    #torch.save(class_b_agents[0].state_dict(), os.path.join(model_dir, f"actor_class_B.pth"))


    # # Step 1: Add agent class identifier into observation
    # for agent, obs in observations.items():
    #     # If the agent is an adversary (either Class A or Class B)
    #     if agent in class_a_agents or agent in class_b_agents:
    #         agent_class = 0 if agent in class_a_agents else 1  # Class A: 0, Class B: 1
    #         updated_obs = np.concatenate([obs, [agent_class]])  # Add class identifier
    #     else:
    #         updated_obs = obs  # Non-adversary agents keep their observation
    #     updated_observations[agent] = updated_obs

save_models(num_class_a, num_class_b, adversary_agents)

# Save Actor networks
#for i in range(1):
#    torch.save(actor_networks[agent].state_dict(), os.path.join(model_dir, f"actor_{agent}.pth"))

# Save GCN model
torch.save(gcn_layer.state_dict(), os.path.join(model_dir, "gcn_model.pth"))

print("Models saved successfully.")


ctype:['adversary_0', 'adversary_1', 'adversary_2']
ctype:['adversary_3']
agent A:adversary_0
agent B:adversary_3
Models saved successfully.


### Load the Models and Test with Different Number of Adversaries

In [10]:
# Set up the environment with a different number of adversaries
new_num_adversaries = 6  # Change the number of adversaries
env = simple_tag_v2.parallel_env(render_mode=None, num_adversaries=new_num_adversaries, num_good=1, num_obstacles=2)
env.reset()

# Update adversary agents list
adversary_agents = [agent for agent in env.agents if 'adversary' in agent]
good_agents = [agent for agent in env.agents if 'agent' in agent]

# Re-initialize embedding and GCN layers
embedding_layer = LinearEmbedding(input_dim=5, embed_dim=embedding_dim)  # Same as before
gcn_layer = GCNLayer(input_dim=embedding_dim, output_dim=gcn_output_dim)
gcn_layer.load_state_dict(torch.load(os.path.join(model_dir, "gcn_model.pth")))

class_a_agents = ['adversary_0', 'adversary_1', 'adversary_2', 'adversary_3']
class_b_agents = ['adversary_4', 'adversary_5']

# Re-initialize Actor networks for new agents and load the saved models
actor_networks = {}
for agent in adversary_agents:
    print(f"Agent: {agent}")
    input_dim = input_actor_network_max_dim #len(env.observation_space(agent).low) + 1 + gcn_output_dim  # Adjust if observation space changes
    output_dim = env.action_space(agent).n  # Assuming discrete action space
    print(f'agent:{agent} input_dim:{input_dim} output_dim:{output_dim}')
    actor_net = ActorNetwork(input_dim, output_dim)
    # Load the saved model (using the first saved model for simplicity)
    if agent in class_a_agents:
        print(f"Agent: {agent} class A")
        actor_net.load_state_dict(torch.load(os.path.join(model_dir, f"actor_class_A.pth")))
        actor_networks[agent] = actor_net
    if agent in class_b_agents:
        print(f"Agent: {agent} class B")
        actor_net.load_state_dict(torch.load(os.path.join(model_dir, f"actor_class_B.pth")))
        actor_networks[agent] = actor_net

    # Adjust the logic as per your agent classes
    #saved_agent = 'adversary_0' if 'adversary_0' in agent or 'adversary_1' in agent or 'adversary_2' in agent else 'adversary_3'
    #actor_net.load_state_dict(torch.load(os.path.join(model_dir, f"actor_{saved_agent}.pth")))
    #actor_networks[agent] = actor_net

print("Models loaded successfully.")


Agent: adversary_0
agent:adversary_0 input_dim:40 output_dim:5
Agent: adversary_0 class A
Agent: adversary_1
agent:adversary_1 input_dim:40 output_dim:5
Agent: adversary_1 class A
Agent: adversary_2
agent:adversary_2 input_dim:40 output_dim:5
Agent: adversary_2 class A
Agent: adversary_3
agent:adversary_3 input_dim:40 output_dim:5
Agent: adversary_3 class A
Agent: adversary_4
agent:adversary_4 input_dim:40 output_dim:5
Agent: adversary_4 class B
Agent: adversary_5
agent:adversary_5 input_dim:40 output_dim:5
Agent: adversary_5 class B
Models loaded successfully.


/tmp/ipykernel_641185/3920034242.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  gcn_layer.load_state_dict(torch.load(os.path.join(model_dir, "gcn_model.pth")))
/tmp/ip

### Evaluation Loop

In [11]:
# Evaluation parameters
num_episodes = 10  # Total number of episodes to run
print_interval = 1  # Print rewards every 10 episodes

# Set models to evaluation mode
for agent in adversary_agents:
    actor_networks[agent].eval()

gcn_layer.eval()  # If GCN layer is being used during evaluation

# Initialize reward tracking
episode_rewards = []
cumulative_reward = 0

# Evaluation loop (no training, deactivate gradient computation)
for episode in range(1, num_episodes + 1):
    observations = env.reset()
    done = False
    cumulative_reward = 0  # Reset cumulative reward for the episode
    
    while not done:
        # Apply the observation wrapper for adversaries
        observations, final_inputs = adversary_observation_wrapper(
            observations, num_class_a, num_class_b, adversary_agents, num_adversaries, num_agents, num_obstacles)
        actions = {}
        
        # Deactivate gradients for evaluation
        with torch.no_grad():
            for agent in env.agents:
                if agent in adversary_agents:
                    #print(f'final_inputs:{final_inputs[agent].shape}')
                    # Padding
                    # m = nn.ConstantPad1d((0, input_actor_network_max_dim - final_inputs[agent].shape[0]), 0)
                    # final_inputs_pad = m(final_inputs[agent])
                    # Seq2Seq
                    # If the input size is different from the hidden size, project the input
                    input_size = 1 
                    input_seq = final_inputs[agent] # expected Batch x Num Sequences x Sequence Length
                    input_seq = input_seq.unsqueeze(1).unsqueeze(0)
                    #print(f'input_seq:{input_seq.shape}')
                    seq_lengths = [final_inputs[agent].shape[0]]
                    if input_size != hidden_size:
                        input_seq = F.linear(input_seq, torch.randn(hidden_size, input_size))

                    # Forward pass
                    final_inputs_pad = model(input_seq, seq_lengths, output_length)
                    #print(f'final_inputs_pad:{final_inputs_pad.shape}')
                    final_inputs_pad = final_inputs_pad.squeeze(0).squeeze(1)
                    #print(f'final_inputs_pad:{final_inputs_pad.shape}')

                    # Get the input for the Actor network
                    actor_input = final_inputs_pad
                    # Get action probabilities (assuming discrete action space)
                    action_probs = actor_networks[agent](actor_input)
                    # Sample an action (highest probability)
                    action = torch.argmax(action_probs).item()
                    actions[agent] = action
                else:
                    # For non-adversary agents, sample random actions
                    actions[agent] = env.action_space(agent).sample()
        
        # Step the environment
        next_observations, rewards, terminations, truncations, infos = env.step(actions)
        
        # Update cumulative reward
        cumulative_reward += sum(rewards.values())
        
        # Update observations
        observations = next_observations
        
        # Check if all agents are done
        done = all(terminations.values()) or all(truncations.values())
    
    # Append cumulative reward for the episode
    episode_rewards.append(cumulative_reward)
    
    # Print rewards every 'print_interval' episodes
    if episode % print_interval == 0:
        avg_reward = sum(episode_rewards[-print_interval:]) / print_interval
        print(f"Episode {episode}: Average Reward: {avg_reward}")


Episode 1: Average Reward: 0.0
Episode 2: Average Reward: -47.2329169072379
Episode 3: Average Reward: 0.0
Episode 4: Average Reward: -2.86485762715734
Episode 5: Average Reward: 0.0
Episode 6: Average Reward: 0.0
Episode 7: Average Reward: 0.0
Episode 8: Average Reward: 0.0
Episode 9: Average Reward: -15.868164298786963
Episode 10: Average Reward: 0.0
